# Preprocesamiento del conjunto de datos de enfermedades cardíacas

En este cuaderno se preparan los datos para el modelado reproduciendo el procedimiento del artículo semilla (Honi y Szathmary, 2024): se conservan los 1025 registros, se divide 70 % entrenamiento y 30 % prueba, y se reescalan las 13 variables al rango [0, 1]. El artículo no indica la semilla ni la estratificación de la división; esas decisiones son propias de este trabajo.

## 1. Carga del dataset original

Se importan las librerías, se lee `heart.csv` y se guarda una huella digital (SHA-256) del archivo para comprobar al final que no fue modificado.

In [1]:
import hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

ruta = "../data/heart.csv"

def huella(ruta_archivo):
    with open(ruta_archivo, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

huella_inicial = huella(ruta)
df = pd.read_csv(ruta)

print("Dimensiones:", df.shape)
print("Huella inicial:", huella_inicial)

Dimensiones: (1025, 14)
Huella inicial: ddb2996b2f4db2e00aad13f4518200179ff69f79093838e3c21ffa672ebec0f1


**Interpretación del resultado**

El dataset original se cargó con 1025 filas y 14 columnas, igual que en el Notebook 01. La huella SHA-256 identifica de forma única el contenido del archivo; al final se calculará de nuevo para demostrar que `heart.csv` permaneció intacto.

## 2. Copia de trabajo

Se crea una copia de `df` para trabajar sobre ella y dejar el original sin tocar.

In [2]:
df_trabajo = df.copy()

print("Copia idéntica al original:", df_trabajo.equals(df))
print("Son el mismo objeto en memoria:", df_trabajo is df)

Copia idéntica al original: True
Son el mismo objeto en memoria: False


**Interpretación del resultado**

La copia tiene el mismo contenido que el original, pero es un objeto distinto en memoria. Cualquier cambio posterior afectaría solo a `df_trabajo`, nunca a `df` ni al archivo en disco.

## 3. Revisión de duplicados

Se mide cuántas filas están repetidas y cuántas veces aparece cada fila distinta.

In [3]:
n_original = len(df_trabajo)
n_duplicados = df_trabajo.duplicated().sum()
repeticiones = df_trabajo.value_counts()

print("Registros originales:", n_original)
print("Filas duplicadas exactas:", n_duplicados)
print("Filas distintas:", len(repeticiones))
print()
print("Veces que aparece cada fila distinta:")
print(repeticiones.describe().round(2))

Registros originales: 1025
Filas duplicadas exactas: 723
Filas distintas: 302

Veces que aparece cada fila distinta:
count    302.00
mean       3.39
std        0.55
min        3.00
25%        3.00
50%        3.00
75%        4.00
max        8.00
Name: count, dtype: float64


**Interpretación del resultado**

De los 1025 registros, 723 son copias adicionales y solo 302 filas son distintas. Cada fila distinta aparece entre 3 y 8 veces (3.39 en promedio), así que todo el conjunto está formado por repeticiones. La información independiente real equivale a 302 registros, no a 1025.

## 4. Decisión sobre los duplicados en el flujo principal

El flujo principal conserva los 1025 registros porque el objetivo es reproducir el artículo, que trabaja con 1025 pacientes y no menciona los duplicados.

In [4]:
df_principal = df_trabajo

print("Registros en el flujo principal:", len(df_principal))
print("Columnas en el flujo principal:", df_principal.shape[1])
print("Duplicados conservados:", df_principal.duplicated().sum())

Registros en el flujo principal: 1025
Columnas en el flujo principal: 14
Duplicados conservados: 723


**Interpretación del resultado**

El flujo principal usa los 1025 registros y las 14 columnas, con sus 723 duplicados. Esto es una **limitación metodológica**: al dividir al azar, una misma fila puede quedar en entrenamiento y en prueba, lo que es una posible fuente de fuga de información y puede inflar las métricas. Se acepta para mantener la correspondencia con el artículo, y la magnitud del problema se mide más adelante.

## 5. Separación de variables predictoras y variable objetivo

Se separa la tabla en `X` (las 13 variables del paciente) e `y` (`target`, lo que se quiere predecir).

In [5]:
X = df_principal.drop(columns="target")
y = df_principal["target"]

print("X:", X.shape)
print("y:", y.shape)
print("Columnas de X:", list(X.columns))
print("target está en X:", "target" in X.columns)

X: (1025, 13)
y: (1025,)
Columnas de X: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
target está en X: False


**Interpretación del resultado**

`X` tiene 1025 filas y 13 columnas, e `y` tiene 1025 valores. `target` no aparece en `X`, de modo que el modelo no recibirá la respuesta como dato de entrada.

## 6. Identificación de variables numéricas y categóricas

Se separan las variables continuas de las categóricas codificadas con números, porque no significan lo mismo aunque ambas sean números.

In [6]:
numericas = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categoricas = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

print("Numéricas:", numericas)
print("Categóricas:", categoricas)
print("Todas las columnas están clasificadas:", sorted(numericas + categoricas) == sorted(X.columns))
print()
print("Valores distintos por variable:")
print(X.nunique())

Numéricas: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categóricas: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Todas las columnas están clasificadas: True

Valores distintos por variable:
age          41
sex           2
cp            4
trestbps     49
chol        152
fbs           2
restecg       3
thalach      91
exang         2
oldpeak      40
slope         3
ca            5
thal          4
dtype: int64


**Interpretación del resultado**

Las 5 variables numéricas toman muchos valores distintos (por ejemplo `chol` tiene 152 y `thalach` 91) porque miden cantidades. Las 8 categóricas toman entre 2 y 5 valores que funcionan como códigos: en `cp`, un 3 no significa tres veces más que un 1. Esta distinción importa al decidir qué transformación recibe cada grupo.

## 7. División en entrenamiento y prueba

Se divide 70 % para entrenamiento y 30 % para prueba, antes de ajustar cualquier transformación, para evitar fuga de información.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)
print()
print("target en entrenamiento:", y_train.value_counts().sort_index().to_dict())
print("target en prueba:       ", y_test.value_counts().sort_index().to_dict())

X_train: (717, 13) | y_train: (717,)
X_test:  (308, 13) | y_test:  (308,)

target en entrenamiento: {0: 349, 1: 368}
target en prueba:        {0: 150, 1: 158}


**Interpretación del resultado**

Quedan 717 registros para entrenamiento y 308 para prueba, es decir, 70.0 % y 30.0 %. La estratificación mantiene la proporción de clases en ambos conjuntos: 349 y 368 en entrenamiento, 150 y 158 en prueba. La proporción 70/30 proviene del artículo; `random_state=42` y `stratify=y` son decisiones propias.

## 8. Codificación de variables categóricas

Se revisa cuántas columnas resultarían con One-Hot Encoding para decidir si se aplica, sin aplicarlo.

In [8]:
columnas_one_hot = sum(X_train[c].nunique() for c in categoricas)
columnas_totales = len(numericas) + columnas_one_hot

print("Columnas de X actualmente:", X_train.shape[1])
print("Columnas resultantes con One-Hot en las categóricas:", columnas_totales)
print("Entrada de la CNN indicada en la Tabla 2 del artículo: (13, 1)")

Columnas de X actualmente: 13
Columnas resultantes con One-Hot en las categóricas: 30
Entrada de la CNN indicada en la Tabla 2 del artículo: (13, 1)


**Interpretación del resultado**

Con One-Hot las 13 columnas pasarían a 30, y la entrada ya no coincidiría con la forma (13, 1) que el artículo indica para su CNN. Aunque el texto del artículo menciona One-Hot, el artículo no explica esa diferencia. **No se aplica One-Hot** en este notebook y se mantienen las 13 variables originales; podrá probarse como variante en una etapa posterior.

## 9. Normalización de las variables

Se reescalan las variables al rango [0, 1] con MinMaxScaler, ajustándolo solo con los datos de entrenamiento y aplicándolo luego a entrenamiento y prueba.

In [9]:
escalador = MinMaxScaler()
escalador.fit(X_train)

X_train_esc = pd.DataFrame(escalador.transform(X_train), columns=X.columns, index=X_train.index)
X_test_esc = pd.DataFrame(escalador.transform(X_test), columns=X.columns, index=X_test.index)

print("Filas usadas para ajustar el escalador:", escalador.n_samples_seen_)
print()
print(pd.DataFrame({"min": escalador.data_min_, "max": escalador.data_max_}, index=X.columns))

Filas usadas para ajustar el escalador: 717

            min    max
age        29.0   77.0
sex         0.0    1.0
cp          0.0    3.0
trestbps   94.0  200.0
chol      126.0  564.0
fbs         0.0    1.0
restecg     0.0    2.0
thalach    71.0  202.0
exang       0.0    1.0
oldpeak     0.0    6.2
slope       0.0    2.0
ca          0.0    4.0
thal        0.0    3.0


**Interpretación del resultado**

El escalador se ajustó con 717 filas, las de entrenamiento, y aprendió el mínimo y el máximo de cada columna con la fórmula del artículo (X - Xmin) / (Xmax - Xmin). Después esos mismos valores se aplicaron a entrenamiento y a prueba, de modo que la prueba no influye en los parámetros. El artículo escala todos los atributos, por eso se incluyen también las categóricas codificadas.

## 10. Verificación del conjunto de entrenamiento

Se comprueba que el conjunto de entrenamiento escalado tenga las dimensiones esperadas, no tenga nulos y quede dentro de [0, 1].

In [10]:
print("X_train escalado:", X_train_esc.shape)
print("y_train:", y_train.shape)
print("Valores nulos:", int(X_train_esc.isnull().sum().sum()))
print("Mínimo global:", X_train_esc.min().min())
print("Máximo global:", X_train_esc.max().max())
print()
X_train_esc.describe().loc[["min", "max", "mean"]].round(3)

X_train escalado: (717, 13)
y_train: (717,)
Valores nulos: 0
Mínimo global: 0.0
Máximo global: 1.0



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
min,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,0.00
max,1.000,1.0,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.00,1.000,1.00
mean,0.533,0.7,0.312,0.359,0.281,0.151,0.258,0.599,0.333,0.173,0.69,0.188,0.78


**Interpretación del resultado**

El conjunto de entrenamiento tiene 717 filas y 13 columnas, sin nulos, con mínimo global 0 y máximo global 1. Es lo esperado: el escalador se ajustó con estos mismos datos, por lo que cada columna ocupa exactamente todo el rango [0, 1].

## 11. Verificación del conjunto de prueba

Se repiten las comprobaciones en el conjunto de prueba y se mide cuántas de sus filas también están en entrenamiento.

In [11]:
print("X_test escalado:", X_test_esc.shape)
print("y_test:", y_test.shape)
print("Valores nulos:", int(X_test_esc.isnull().sum().sum()))
print("Mínimo global:", X_test_esc.min().min())
print("Máximo global:", X_test_esc.max().max())

fuera = X_test_esc.columns[(X_test_esc.min() < 0) | (X_test_esc.max() > 1)].tolist()
print("Columnas fuera de [0, 1]:", fuera)

filas_entrenamiento = set(map(tuple, X_train.assign(target=y_train).values))
repetidas = sum(tuple(f) in filas_entrenamiento for f in X_test.assign(target=y_test).values)
print()
print("Filas de prueba:", len(X_test))
print("Filas de prueba que también están en entrenamiento:", repetidas)
print("Porcentaje: {:.1f}%".format(100 * repetidas / len(X_test)))

X_test escalado: (308, 13)
y_test: (308,)
Valores nulos: 0
Mínimo global: 0.0
Máximo global: 1.0
Columnas fuera de [0, 1]: []

Filas de prueba: 308
Filas de prueba que también están en entrenamiento: 286
Porcentaje: 92.9%


**Interpretación del resultado**

El conjunto de prueba tiene 308 filas y 13 columnas, sin nulos, y todos sus valores quedan entre 0 y 1. En general un valor de prueba puede superar 1 si excede el máximo visto en entrenamiento, y no se corregiría, pero aquí no ocurrió, probablemente porque cada fila se repite varias veces y los valores extremos también aparecen en entrenamiento. Además, 286 de las 308 filas de prueba (92.9 %) tienen una copia exacta en entrenamiento: esa es la magnitud de la fuga de información que se acepta en el flujo principal.

## 12. Comprobación del archivo original y guardado

Se verifica que `heart.csv` no cambió y se guardan los cuatro conjuntos procesados en `resultados/csv` para el notebook siguiente.

In [12]:
print("Archivo original sin cambios:", huella(ruta) == huella_inicial)

carpeta = "../resultados/csv/"
X_train_esc.to_csv(carpeta + "X_train.csv", index=False)
X_test_esc.to_csv(carpeta + "X_test.csv", index=False)
y_train.to_csv(carpeta + "y_train.csv", index=False)
y_test.to_csv(carpeta + "y_test.csv", index=False)

for nombre in ["X_train", "X_test", "y_train", "y_test"]:
    print(nombre, pd.read_csv(carpeta + nombre + ".csv").shape)

Archivo original sin cambios: True
X_train (717, 13)
X_test (308, 13)
y_train (717, 1)


y_test (308, 1)


**Interpretación del resultado**

La huella del archivo coincide con la inicial, así que `heart.csv` quedó intacto. Los cuatro conjuntos se guardaron y, al leerlos de nuevo, conservan las dimensiones (717, 13), (308, 13), (717, 1) y (308, 1). Contienen los datos del flujo principal ya escalados con el escalador ajustado solo con entrenamiento.

## Análisis adicional: efecto de eliminar duplicados

Se calcula qué pasaría si se eliminaran los duplicados. Esta variante no se usa en el flujo principal ni modifica los conjuntos guardados.

In [13]:
df_sin_dup = df_trabajo.drop_duplicates().reset_index(drop=True)
X_sd = df_sin_dup.drop(columns="target")
y_sd = df_sin_dup["target"]

Xtr_sd, Xte_sd, ytr_sd, yte_sd = train_test_split(
    X_sd, y_sd, test_size=0.30, random_state=42, stratify=y_sd
)

print("Registros sin duplicados:", len(df_sin_dup))
print("target sin duplicados:", y_sd.value_counts().sort_index().to_dict())
print("Entrenamiento / prueba:", len(Xtr_sd), "/", len(Xte_sd))

filas_tr_sd = set(map(tuple, Xtr_sd.assign(target=ytr_sd).values))
repetidas_sd = sum(tuple(f) in filas_tr_sd for f in Xte_sd.assign(target=yte_sd).values)
print("Filas de prueba presentes en entrenamiento:", repetidas_sd)

print()
print("Flujo principal: X_train", X_train.shape, "| X_test", X_test.shape)

Registros sin duplicados: 302
target sin duplicados: {0: 138, 1: 164}
Entrenamiento / prueba: 211 / 91
Filas de prueba presentes en entrenamiento: 0

Flujo principal: X_train (717, 13) | X_test (308, 13)


**Interpretación del resultado**

Al eliminar duplicados quedan 302 registros, con 138 de `target = 0` y 164 de `target = 1`, y una división 70/30 daría 211 para entrenamiento y 91 para prueba, sin ninguna fila de prueba repetida en entrenamiento. Esta variante **no corresponde a la reproducción del artículo**, que usa 1025 pacientes. Queda disponible como análisis de sensibilidad o experimento adicional para comprobar cuánto cambian los resultados sin la fuga de información.

## Resumen del preprocesamiento

| Paso | Decisión | Origen |
|---|---|---|
| Duplicados | Se conservan los 1025 registros (723 duplicados exactos, 302 filas distintas) | Reproducción del artículo; limitación documentada |
| Variable objetivo | `target` separada de las 13 variables predictoras | Estándar |
| División | 70 % entrenamiento (717) y 30 % prueba (308) | Proporción del artículo |
| Semilla y estratificación | `random_state=42`, `stratify=y` | Propias; el artículo no las indica |
| Codificación | No se aplica One-Hot; se mantienen las 13 columnas | Coincidir con la entrada (13, 1) del artículo |
| Normalización | MinMaxScaler a [0, 1] en las 13 variables, ajustado solo con entrenamiento | Artículo (ecuación 1); orden de ajuste según regla anti-fuga |
| Análisis adicional | Variante sin duplicados: 302 registros, 211 / 91 | Solo análisis de sensibilidad |

Dimensiones finales: `X_train` (717, 13), `X_test` (308, 13), `y_train` (717,), `y_test` (308,). No se detectaron valores nulos y `heart.csv` no fue modificado.

## Conclusión de esta fase

Los datos quedaron preparados siguiendo el procedimiento del artículo: 1025 registros divididos en 717 para entrenamiento y 308 para prueba, con las 13 variables escaladas a [0, 1] a partir de parámetros aprendidos solo con entrenamiento. La principal limitación es que el 92.9 % de las filas de prueba también aparece en entrenamiento por los duplicados, lo que podría inflar las métricas del modelo. La variante sin duplicados (302 registros) queda documentada para un análisis posterior. En esta fase no se entrenó ningún modelo; el siguiente paso es el Notebook 03.